# TradeFlow AI — nb3_olm_finetune (v2 — 4-Phase Curriculum)

**Model**: `allenai/olmOCR-7B-0225-preview` + LoRA fine-tuning  
**GPU**: Kaggle T4×2

Ini adalah skrip utama untuk melatih model pembacaan dokumen. Notebook ini sudah diatur dengan path Kaggle *default*, sehingga Anda cukup menekan **Save & Run All (Commit)**.

In [ ]:
!pip install -q peft transformers datasets accelerate bitsandbytes trl
!pip install -q pdf2image pillow albumentations
!apt-get install -qq poppler-utils

In [ ]:
import json, os
import torch
import numpy as np
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

MODEL_ID  = 'allenai/olmOCR-7B-0225-preview'
OUT_DIR   = Path('./olmocr-tradeflow-lora')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── KAGGLE PATHS (PASTIKAN DATASET INI SUDAH DI-ADD KE NOTEBOOK) ──
# Output dari nb0_real_doc_augmentation
NB0_INPUT = Path('/kaggle/input/tradeflow-nb0') 
# Output dari nb1_synthetic_generator
NB1_INPUT = Path('/kaggle/input/tradeflow-nb1')
# Dataset dokumen asli dan ground truth
REAL_DOCS = Path('/kaggle/input/tradeflow-real-docs')

MANIFEST_PATH = NB0_INPUT / 'dataset/augmented_manifest.json'
SYNTHETIC_DIR = NB1_INPUT / 'dataset/synthetic'
GT_PATH       = REAL_DOCS / 'TradeFlow_GroundTruth_v5.2.json'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

In [ ]:
print("=== PREPARING MODEL ===")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Load 4-bit quantization untuk memuat model 7B di Kaggle T4 (16GB VRAM)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    load_in_4bit=True,
    device_map='auto',
    torch_dtype=torch.float16
)
model = prepare_model_for_kbit_training(model)

# LoRA Config sesuai strategi Anti-Memorisasi (Rank 32, Dropout 0.1)
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.10,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
print("Model disiapkan dengan LoRA!")

In [ ]:
print("\n=== PHASE 1: SYNTHETIC SFT (STRATEGY MIX) ===")
print("Karena limitasi waktu Kaggle, kita menggabungkan Phase 1 & 2 ke dalam single mixed-dataset.")

# Load JSON sintetik dari nb1
texts = []
if SYNTHETIC_DIR.exists():
    for f in os.listdir(SYNTHETIC_DIR):
        if f.endswith('.json'):
            with open(SYNTHETIC_DIR / f, 'r') as file:
                data = file.read()
                prompt = f"Extract fields from this document:\n```json\n{data}\n```"
                texts.append(prompt)
else:
    print(f"[WARNING] Folder sintetik {SYNTHETIC_DIR} tidak ditemukan. Memakai data dummy untuk testing pipeline...")
    texts = ["Extract fields: B/L 12345", "Extract fields: B/L 67890"]

train_dataset = Dataset.from_dict({'text': texts})

training_args = TrainingArguments(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=50, # Setel ke angka lebih besar (misal 500) untuk full training
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    optim="paged_adamw_8bit",
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    label_smoothing_factor=0.1, # Sesuai strategi nb3
    save_strategy="no" # Simpan manual nanti
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=512
)

print("Memulai Training...")
trainer.train()

print("\nMenyimpan model terbaik...")
model.save_pretrained(OUT_DIR / "best")
tokenizer.save_pretrained(OUT_DIR / "best")
print(f"Model tersimpan di {OUT_DIR / 'best'}")

In [ ]:
print("\n=== UPLOAD KE HUGGINGFACE ===")
HF_REPO_NAME = 'muhammadghiffari/olm-ocr-tradeflow-lora'  # GANTI INI DENGAN REPO ANDA
HF_TOKEN = os.environ.get('HF_TOKEN') # Pastikan Anda memasukkan Kaggle Secrets bernama HF_TOKEN

if HF_TOKEN:
    print(f"Uploading model to {HF_REPO_NAME}...")
    model.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
    print("✅ Upload Berhasil! Silakan update .env Docker lokal Anda dengan LORA_ADAPTER_ID ini.")
else:
    print("❌ HF_TOKEN tidak ditemukan di Kaggle Secrets. Melewati proses upload.")
    print("Anda bisa mengunduh file secara manual dari folder output Kaggle.")
